# 4.7 — Topographic Controls on Prediction Performance

Elevation, relief, TPI and the 4-class terrain classification vs MAE.
Focus on **v27 at MR=0.5, masked stations only** at Δ ≈ 3 h.

This is the cleanest test of: *what makes a station difficult to
reconstruct when it is missing?*

**Caveat:** because the v27 mask is fixed, the masked stations are a
particular subset of the network. Correlations are conditional on that
subset (N_masked stations).

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS)
print("Models at MR=0.00:", MR0_RUNS)

MR5_RUNS = [r for r in RUNS if "mr0.50" in RUNS[r]]
AGG5 = {r: C.load_agg(r, "mr0.50") for r in MR5_RUNS}
TC_IDX = C.terrain_class_indices(stn)
REF_KI = 6  # ≈ 3 h

# NOTE (2026-09-01): the MR=0.5 evaluation mask is NOT fixed across windows.
# predictions.pt has 11,684 distinct masks (77 stations each); every station is
# masked in ~48-51% of windows. mod_msk_* / mod_vis_* therefore exist for ALL
# 155 stations, and each station has both a masked-window and a visible-window
# MAE. Plot both for every station instead of splitting stations by MI[0].
# is_masked below is the FIRST window's mask only; kept for the map cell (p4).
import torch
d5 = C.load_dump("v27", "mr0.50")
MI = d5["masked_idx"]
masked_set = set(MI[0].tolist())
is_masked = np.array([i in masked_set for i in range(len(stn))])
N_masked = is_masked.sum()
print(f"Masked stations: {N_masked} / {len(stn)}")
del d5

In [ ]:
# ── Marker convention across all MR=0.5 maps ──
# masked stations  = triangle marker
# visible stations = 'o' marker
MRK_MASKED, MRK_VISIBLE = "^", "o"
COL_MASKED, COL_VISIBLE = "#C4502A", "#1F5F6B"

In [ ]:
import geopandas as gpd
import rioxarray  # noqa

PROJ = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) \
       if os.path.isfile("common.py") else os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

PATH_SWISSSHAPE = os.path.expanduser(
    os.environ.get("SWISSSHAPE",
        os.path.join(PROJ, "swissboundaries3d_2056.shp.zip")))
_CAND = [os.environ.get("DATA_ROOT", ""),
         os.path.expanduser("~/PeakWeatherDataset"),
         os.path.join(PROJ, "PeakWeatherDataset")]
DATA_ROOT = next((p for p in _CAND if os.path.isdir(str(p))), _CAND[-1])

from peakweather.dataset import PeakWeatherDataset
ds_topo = PeakWeatherDataset(
    root=DATA_ROOT,
    parameters=["temperature", "pressure", "humidity",
                 "wind_speed", "wind_direction", "precipitation"],
    compute_uv=True, station_type="meteo_station",
    imputation_method=None, freq="d", extended_topo_vars="DEM")

def _load_dem_and_border(ds_topo, path_swissshape, coarsen=10):
    switzerland = gpd.read_file(
        path_swissshape,
        layer='swissBOUNDARIES3D_1_5_TLM_LANDESGEBIET').to_crs('EPSG:2056')
    minx, miny, maxx, maxy = switzerland.total_bounds
    topo = ds_topo.load_topography()
    dem  = topo['topo_DEM'].dem
    dem_ch = dem.rio.clip(switzerland.geometry, switzerland.crs, drop=False)
    dem_bg = dem.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_fg = dem_ch.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_bg = dem_bg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    dem_fg = dem_fg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    return dem_bg, dem_fg, switzerland

def draw_dem(ax, dem_bg, dem_fg, switzerland):
    norm = mcolors.Normalize(vmin=0, vmax=4500)
    dem_bg.plot(ax=ax, cmap='terrain', norm=norm, alpha=0.35,
                robust=True, add_labels=False, add_colorbar=False)
    dem_fg.plot(ax=ax, cmap='terrain', norm=norm,
                robust=True, add_labels=False, add_colorbar=False)
    switzerland.boundary.plot(ax=ax, color='white', linewidth=1.0)
    ax.axis('off')

print('Loading DEM + border ...')
dem_bg, dem_fg, switzerland = _load_dem_and_border(ds_topo, PATH_SWISSSHAPE)
print('Done.')

## Exclusion overlay

Every per-station scatter map below also marks stations excluded for that
variable (`common.excluded_station_variable_reasons()`) with an X:
**black** = missing in both train and test, **white** = missing in train
only, **grey** = missing in test only, **orange** = present in both splits
but excluded for a data-quality reason (BIZ/pressure, sensor drift).

In [ ]:
REASON_STYLE = {
    "missing_train": dict(color="white",   label="missing in train only"),
    "missing_test":  dict(color="0.6",     label="missing in test only"),
    "missing_both":  dict(color="black",   label="missing in train & test"),
    "drift":         dict(color="#E8A838", label="excluded \u2014 sensor drift"),
}
EXCL_REASONS = C.excluded_station_variable_reasons()

def overlay_exclusion_markers(ax, variable, s=70, legend=False):
    return  # X markers removed per user request

## Station map by terrain class on the DEM (report Figure: data_station_map_terrain)

All 155 stations as equal-size circles coloured by the four terrain classes of
`common.station_table()`, over the DEM + Swiss border. Right panel: elevation
distribution stacked by class with the 900 m valley-floor / elevated-enclosed split.
Copy `47_station_map_terrain_dem.png` to the report as `figures/data_station_map_terrain.png`.


## MAE vs elevation — scatter

Each point is one masked station. Pearson r reported.

In [ ]:
# ── Station map by terrain class on the DEM + elevation histogram ────────────
TC_COLORS_MAP = {"valley floor": "#2B7A78", "elevated enclosed": "#D4A373",
                 "exposed ridge/summit": "#BC4749", "open / slope": "#5B8E7D"}
h_all = stn.height.values.astype(float)
tc_all = stn.terrain_class.values

fig, (ax_map, ax_hist) = plt.subplots(
    1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [2.4, 1]})

# Left: DEM background + one equal-size circle per station
draw_dem(ax_map, dem_bg, dem_fg, switzerland)
for tc in C.TCLASSES:
    idx = TC_IDX[tc]
    ax_map.scatter(stn.easting.values[idx], stn.northing.values[idx],
                   s=42, c=TC_COLORS_MAP[tc], marker="o",
                   edgecolors="k", linewidths=0.5, zorder=5,
                   label=f"{tc} (n={len(idx)})")
ax_map.legend(fontsize=8, loc="lower right", framealpha=0.9, title="terrain class",
              title_fontsize=8)
ax_map.set_title(f"{len(stn)} stations by terrain class", fontsize=11)

# Right: elevation histogram stacked by class
bins = np.arange(200, 3700, 200)
bottom = np.zeros(len(bins) - 1)
for tc in C.TCLASSES:
    cnt, _ = np.histogram(h_all[TC_IDX[tc]], bins=bins)
    ax_hist.bar(bins[:-1], cnt, width=200, bottom=bottom, align="edge",
                color=TC_COLORS_MAP[tc], edgecolor="k", linewidth=0.3)
    bottom += cnt
ax_hist.axvline(900, color="k", ls="--", lw=0.9)
ax_hist.text(930, ax_hist.get_ylim()[1] * 0.95, "900 m", fontsize=8, va="top")
ax_hist.set_xlabel("Station elevation [m]"); ax_hist.set_ylabel("# stations")
ax_hist.set_title("Elevation distribution", fontsize=11); ax_hist.grid(alpha=0.25, axis="y")

plt.tight_layout()
C.save_fig(fig, "47_station_map_terrain_dem")
plt.show(); plt.close(fig)


In [ ]:
# ── Station map by terrain class on the DEM + elevation histogram ────────────
TC_COLORS_MAP = {"valley floor": "#2B7A78", "elevated enclosed": "#D4A373",
                 "exposed ridge/summit": "#BC4749", "open / slope": "#5B8E7D"}
h_all = stn.height.values.astype(float)
tc_all = stn.terrain_class.values

fig, ax_map = plt.subplots(figsize=(14, 5.2))

# Left: DEM background + one equal-size circle per station
draw_dem(ax_map, dem_bg, dem_fg, switzerland)
for tc in C.TCLASSES:
    idx = TC_IDX[tc]
    ax_map.scatter(stn.easting.values[idx], stn.northing.values[idx],
                   s=42, c=TC_COLORS_MAP[tc], marker="o",
                   edgecolors="k", linewidths=0.5, zorder=5,
                   label=f"{tc} (n={len(idx)})")
ax_map.legend(fontsize=8, loc="lower right", framealpha=0.9, title="terrain class",
              title_fontsize=8)
ax_map.set_title(f"{len(stn)} stations by terrain class", fontsize=11)


plt.tight_layout()
C.save_fig(fig, "47_station_map_terrain_dem_v2")
plt.show(); plt.close(fig)


In [ ]:
from scipy.stats import pearsonr
fig, axes = plt.subplots(1, NV, figsize=(17, 3.5))
a = AGG5["v27"]
h = stn.height.values
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    # Masked stations
    cnt_m = a["mod_msk_cnt"][REF_KI, :, vi]
    s_m   = a["mod_msk_sum_phys"][REF_KI, :, vi]
    mae_m = np.where(cnt_m > 0, s_m / np.maximum(cnt_m, 1), np.nan)
    valid_m = ~np.isnan(mae_m)
    ax.scatter(h[valid_m], mae_m[valid_m], s=20, alpha=0.6,
               color=COL_MASKED, marker=MRK_MASKED, label="masked windows")
    r_m, p_m = pearsonr(h[valid_m], mae_m[valid_m])

    # Visible stations
    cnt_v = a["mod_vis_cnt"][REF_KI, :, vi]
    s_v   = a["mod_vis_sum_phys"][REF_KI, :, vi]
    mae_v = np.where(cnt_v > 0, s_v / np.maximum(cnt_v, 1), np.nan)
    valid_v = ~np.isnan(mae_v)
    ax.scatter(h[valid_v], mae_v[valid_v], s=20, alpha=0.6,
               color=COL_VISIBLE, marker=MRK_VISIBLE, label="visible windows")
    r_v, p_v = pearsonr(h[valid_v], mae_v[valid_v])

    ax.set_title(f"{v}\nmasked r={r_m:.2f} (p={p_m:.1e})\n"
                 f"visible r={r_v:.2f} (p={p_v:.1e})", fontsize=8)
    ax.set_xlabel("Elevation [m]", fontsize=8); ax.grid(alpha=.3)
axes[0].set_ylabel(f"MAE at {LEAD[REF_KI]} (MAE Transformer, MR=0.5)")
axes[-1].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Per-station MAE vs elevation — MAE Transformer, masked vs visible windows, all stations, "
             f"MR=0.5 at {LEAD[REF_KI]}", y=1.06)
plt.tight_layout(); C.save_fig(fig, "47_elevation_scatter"); plt.show()


In [ ]:
from scipy.stats import pearsonr
fig, axes = plt.subplots(1, NV, figsize=(17, 3.5))
a = AGG5["v27"]
tpi = stn.tpi_n10.values
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    # Masked stations
    cnt_m = a["mod_msk_cnt"][REF_KI, :, vi]
    s_m   = a["mod_msk_sum_phys"][REF_KI, :, vi]
    mae_m = np.full(cnt_m.shape, np.nan)
    ok_m = cnt_m > 0
    mae_m[ok_m] = s_m[ok_m] / cnt_m[ok_m]
    valid_m = np.isfinite(mae_m)
    ax.scatter(tpi[valid_m], mae_m[valid_m], s=20, alpha=0.6,
               color=COL_MASKED, marker=MRK_MASKED, label="masked windows")
    r_m, p_m = pearsonr(tpi[valid_m], mae_m[valid_m])

    # Visible stations
    cnt_v = a["mod_vis_cnt"][REF_KI, :, vi]
    s_v   = a["mod_vis_sum_phys"][REF_KI, :, vi]
    mae_v = np.full(cnt_v.shape, np.nan)
    ok_v = cnt_v > 0
    mae_v[ok_v] = s_v[ok_v] / cnt_v[ok_v]
    valid_v = np.isfinite(mae_v)
    ax.scatter(tpi[valid_v], mae_v[valid_v], s=20, alpha=0.6,
               color=COL_VISIBLE, marker=MRK_VISIBLE, label="visible windows")
    r_v, p_v = pearsonr(tpi[valid_v], mae_v[valid_v])

    ax.set_title(f"{v}\nmasked r={r_m:.2f} (p={p_m:.1e})\n"
                 f"visible r={r_v:.2f} (p={p_v:.1e})", fontsize=8)
    ax.set_xlabel("TPI 10 km (normalised)", fontsize=8); ax.grid(alpha=.3)
axes[0].set_ylabel(f"MAE at {LEAD[REF_KI]} (MAE Transformer, MR=0.5)")
axes[-1].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Per-station MAE vs TPI (10 km) — MAE Transformer, masked vs visible windows, all stations, "
             f"MR=0.5 at {LEAD[REF_KI]}", y=1.06)
plt.tight_layout(); C.save_fig(fig, "47_tpi10_scatter"); plt.show()

## MAE by terrain class — box plot (masked stations only)

In [ ]:
fig, axes = plt.subplots(1, NV, figsize=(17, 3.8))
a = AGG5["v27"]
n_tc = len(C.TCLASSES)
x = np.arange(n_tc)
w = 0.32

for vi, (ax, v) in enumerate(zip(axes, VARS)):
    cnt_m = a["mod_msk_cnt"][REF_KI, :, vi]
    s_m   = a["mod_msk_sum_phys"][REF_KI, :, vi]
    mae_m = np.where(cnt_m > 0, s_m / np.maximum(cnt_m, 1), np.nan)
    cnt_v = a["mod_vis_cnt"][REF_KI, :, vi]
    s_v   = a["mod_vis_sum_phys"][REF_KI, :, vi]
    mae_v = np.where(cnt_v > 0, s_v / np.maximum(cnt_v, 1), np.nan)

    data_m, data_v, labels_tc = [], [], []
    for tc in C.TCLASSES:
        idx = TC_IDX[tc]
        sel_m = idx  # all stations: masked-window MAE
        sel_v = idx  # all stations: visible-window MAE
        vals_m = mae_m[sel_m]; vals_m = vals_m[~np.isnan(vals_m)]
        vals_v = mae_v[sel_v]; vals_v = vals_v[~np.isnan(vals_v)]
        data_m.append(vals_m); data_v.append(vals_v)
        labels_tc.append(f"{tc}\n(n={len(vals_m)})")

    bp_m = ax.boxplot(data_m, positions=x - w / 2 - 0.02, widths=w,
                      patch_artist=True, manage_ticks=False)
    bp_v = ax.boxplot(data_v, positions=x + w / 2 + 0.02, widths=w,
                      patch_artist=True, manage_ticks=False)
    for patch in bp_m["boxes"]:
        patch.set_facecolor(COL_MASKED); patch.set_alpha(0.6)
    for patch in bp_v["boxes"]:
        patch.set_facecolor(COL_VISIBLE); patch.set_alpha(0.6)

    ax.set_xticks(x); ax.set_xticklabels(labels_tc)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    ax.tick_params(axis="x", labelsize=6, rotation=20); ax.grid(alpha=.3)
    if vi == 0:
        ax.legend([bp_m["boxes"][0], bp_v["boxes"][0]], ["masked windows", "visible windows"],
                  fontsize=7, loc="upper left")
axes[0].set_ylabel(f"MAE at {LEAD[REF_KI]}")
fig.suptitle(f"MAE by terrain class — MAE Transformer, masked vs visible windows (MR=0.5) at {LEAD[REF_KI]}", y=1.04)
plt.tight_layout(); C.save_fig(fig, "47_terrain_boxplot"); plt.show()


## Terrain descriptor correlation matrix (masked stations)

Pearson r heatmap — compact summary of all descriptor × variable
correlations. Masked stations only.

In [ ]:
from scipy.stats import pearsonr
topo_cols = ["height", "slope", "relief_2km", "relief_10km",
             "tpi_n2", "tpi_n10", "nn_dist_km", "n_within_50km"]
a = AGG5["v27"]
rows = []
for vi, v in enumerate(VARS):
    cnt = a["mod_msk_cnt"][REF_KI, :, vi]
    s   = a["mod_msk_sum_phys"][REF_KI, :, vi]
    mae = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
    valid = ~np.isnan(mae)  # all stations, masked-window MAE
    for tc in topo_cols:
        r, p = pearsonr(stn[tc].values[valid], mae[valid])
        rows.append({"variable": v, "descriptor": tc, "r": r, "p": p})
corr_df = pd.DataFrame(rows)
piv = corr_df.pivot(index="descriptor", columns="variable", values="r")
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5, aspect="auto")
ax.set_xticks(range(len(piv.columns)))
ax.set_xticklabels(piv.columns, fontsize=9, rotation=30, ha="right")
ax.set_yticks(range(len(piv.index)))
ax.set_yticklabels(piv.index, fontsize=9)
for i in range(len(piv.index)):
    for j in range(len(piv.columns)):
        ax.text(j, i, f"{piv.values[i, j]:.2f}", ha="center", va="center",
                fontsize=7, color="white" if abs(piv.values[i, j]) > 0.3 else "black")
fig.colorbar(im, label="Pearson r")
ax.set_title(f"Topography vs MAE — MAE Transformer, masked windows, all stations, "
             f"MR=0.5 at {LEAD[REF_KI]}")
plt.tight_layout(); C.save_fig(fig, "47_topo_corr_matrix"); plt.show()
display(piv.round(3))

## Per-station MAE map — masked vs visible markers

Masked = ▲, visible = ○. Colour encodes MAE at the reference lead.

In [ ]:
a = AGG5["v27"]
fig, axes = plt.subplots(1, NV, figsize=(6.0 * NV, 7.5))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    cnt_m = a["mod_msk_cnt"][REF_KI, :, vi]
    s_m   = a["mod_msk_sum_phys"][REF_KI, :, vi]
    mae_m = np.where(cnt_m > 0, s_m / np.maximum(cnt_m, 1), np.nan)
    cnt_v = a["mod_vis_cnt"][REF_KI, :, vi]
    s_v   = a["mod_vis_sum_phys"][REF_KI, :, vi]
    mae_v = np.where(cnt_v > 0, s_v / np.maximum(cnt_v, 1), np.nan)
    vmin = np.nanpercentile(np.concatenate([mae_m[is_masked],
                                            mae_v[~is_masked]]), 2)
    vmax = np.nanpercentile(np.concatenate([mae_m[is_masked],
                                            mae_v[~is_masked]]), 98)
    # visible stations
    v_ok = ~is_masked & ~np.isnan(mae_v)
    sc_v = ax.scatter(stn.easting[v_ok], stn.northing[v_ok],
                      c=mae_v[v_ok], s=50, cmap="YlOrRd",
                      marker=MRK_VISIBLE, vmin=vmin, vmax=vmax,
                      edgecolors="k", linewidths=0.3, zorder=5,
                      label="visible")
    # masked stations
    m_ok = is_masked & ~np.isnan(mae_m)
    ax.scatter(stn.easting[m_ok], stn.northing[m_ok],
               c=mae_m[m_ok], s=70, cmap="YlOrRd",
               marker=MRK_MASKED, vmin=vmin, vmax=vmax,
               edgecolors="k", linewidths=0.3, zorder=6,
               label="masked")
    overlay_exclusion_markers(ax, v, legend=False)
    fig.colorbar(sc_v, ax=ax, fraction=0.03, pad=0.02,
                 label=f"MAE [{C.UNITS[v]}]")
    if vi == 0:
        ax.legend(fontsize=9, loc="upper left")
    ax.set_title(f"{v} — MAE Transformer, MR=0.5 at {LEAD[REF_KI]}", fontsize=12)
plt.tight_layout()
C.save_fig(fig, "47_map_msk_side_by_side"); plt.show()

## Interpretation

Positive correlation between elevation/relief and MAE indicates that
complex terrain is harder to forecast/reconstruct. The 4-class terrain
scheme separates valley-floor stations (cold-pool dynamics, fog) from
exposed ridges (synoptic forcing). If enclosed stations show higher
error than open ones at the same height, the enclosure geometry
itself — not just altitude — matters.

All correlations are conditional on the fixed mask subset (N_masked
stations) and should not be overclaimed.

## Stations excluded from evaluation, by variable

Marks every station×variable pair dropped by `DROP_SV` (see common.py) with
an X on the map: **black** = missing in both train and test (>50%),
**white** = missing in train only (sensor added after 2021, present in
test), **grey** = missing in test only (no such pair exists in the current
data — kept for completeness), **orange** = present in both splits but
excluded for a data-quality reason (BIZ/pressure, systematic sensor drift),
not for missingness.

In [ ]:
# ── Stations excluded from evaluation, by variable ──────────────────────────
REASON_STYLE = {
    "missing_train": dict(color="white",   label="missing in train only"),
    "missing_test":  dict(color="0.6",     label="missing in test only"),
    "missing_both":  dict(color="black",   label="missing in train & test"),
    "drift":         dict(color="#E8A838", label="excluded \u2014 sensor drift"),
}
EXCL_REASONS = C.excluded_station_variable_reasons()

fig, axes = plt.subplots(1, NV, figsize=(6.0 * NV, 7.5))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    for reason, style in REASON_STYLE.items():
        idx = [i for i, abbr in enumerate(stn.abbr) if EXCL_REASONS.get((abbr, v)) == reason]
        if not idx:
            continue
        ax.scatter(stn.easting.values[idx], stn.northing.values[idx],
                  marker="X", s=100, c=style["color"], edgecolors="k",
                  linewidths=0.7, zorder=6, label=style["label"])
    ax.set_title(f"{v}", fontsize=12)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(REASON_STYLE),
          fontsize=9, bbox_to_anchor=(0.5, -0.02), frameon=True)
fig.suptitle("Stations excluded from evaluation, by variable", y=1.02)
plt.tight_layout()
C.save_fig(fig, "excluded_stations_map")
plt.show()

print("Excluded station\u00d7variable pairs:")
for (abbr, v), reason in sorted(EXCL_REASONS.items()):
    print(f"  {abbr:<4} {v:<12} {reason}")